In [194]:
import gurobipy as gp
from gurobipy import GRB
from gurobipy import *

In [ ]:
# parameters
#Operations = [1,2,3,4,5,6,7]
Operations = [1, 2, 3]
J= len(Operations)
Ressources = [1,2,3,4,5,6,7]
M= len(Ressources)
T= 16

predecessors = [[],             # Step 1: Station 0_Input
                [0],            # Step 2: Station 1_stearing
                [0],            # Step 2: Station 2_stearing
                [0],            # Step 2: Station 3_stearing
                [1, 2, 3],      # Step 3: Station 4_light
                [1, 2, 3],      # Step 3: Station 5_light
                [1, 2, 3],      # Step 3: Station 6_light
                [4, 5, 6]]      # Step 4: Station 7_Output

#duration = [2, 3, 2, 4, 1, 2, 1]
duration = [2, 3, 2]

#demand_capacity =  [[1, 1, 2, 1, 1, 1, 4],
#                    [2, 1, 1, 2, 1, 1, 3],
#                    [1, 2, 1, 1, 1, 1, 1],
#                    [1, 2, 1, 1, 1, 1, 1],
#                    [1, 2, 3, 1, 3, 1, 2],
#                    [1, 2, 1, 1, 1, 1, 2],
#                    [3, 3, 3, 3, 3, 3, 3]]
demand_capacity= [[1, 1, 2, 1, 1, 1, 4], [2, 1, 1, 2, 1, 1, 3], [1, 2, 1, 1, 1, 1, 1]]

FEZ= [0,0,0,0,0,0,0]
SEZ= [15,15,15,15,15,15,15]

capacity= [10,10,10,10,10,10,10]

In [196]:
var1= [[1,2,3], 5, 7]
var2= [[1,2,3], [4,6], 7]

In [197]:
m = gp.Model("RCPSP")

In [198]:
# decision variables
S = m.addVars(J, T, vtype= GRB.BINARY)
C = m.addVar(lb= 0, vtype= GRB.CONTINUOUS)

In [199]:
m.setObjective(C, GRB.MINIMIZE)

In [200]:
# time constraint
for j in range(J):
    m.addConstr(C >= sum((t + duration[j]) * S[j,t] for t in range(FEZ[j], SEZ[j]+1)))

In [201]:
# timeslot constraint
for j in range(J):
    m.addConstr((quicksum(S[j, t] for t in range(FEZ[j], SEZ[j]+1)) == 1))

In [202]:
# variant based order constraint


In [203]:
# precendence constraint
for j in range(J):
        for h in predecessors[j]:
                m.addConstr(quicksum(t * S[h, t] for t in range(FEZ[h], SEZ[h]+1)) <= quicksum((t - duration[h]) * S[j, t] for t in range(FEZ[j], SEZ[j]+1)))

In [204]:
# capacity constraint
for r in range(M):
    for t in range(T):
        m.addConstr((quicksum(demand_capacity[j][r] * quicksum(S[j,q] for q in range(t, min(t+duration[j], T))) for j in range(J)) <= capacity[r]))

In [205]:
# Solve
m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 120 rows, 49 columns and 908 nonzeros
Model fingerprint: 0x1075e818
Variable types: 1 continuous, 48 integer (48 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+01]
Found heuristic solution: objective 16.0000000
Presolve removed 120 rows and 49 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 12 available processors)

Solution count 2: 5 16 

Optimal solution found (tolerance 1.00e-04)
Best objective 5.000000000000e+00, best bound 5.000000000000e+00, gap 0.0000%


In [206]:
#m.computeIIS()
#m.write("rcpsp.ilp")

In [207]:
print("Objective value: ", m.objVal , " found after ", m.Runtime, " seconds. Relative gap is: ", m.MIPGap)

Objective value:  5.0  found after  0.012000083923339844  seconds. Relative gap is:  0.0
